### Catastrophe Historical Aggregation Pipeline

Creates (or updates) a serverless Lakeflow Declarative Pipeline that rolls the
`{CATALOG}.orders.bronze_hist_*` Delta tables (generated by the
`Catastrophe_History` stage) into Silver → Gold in `{CATALOG}.orders`.

Runs the pipeline once (triggered) and waits for completion, so the gold
analytics tables exist by the time this stage finishes. Depends on
`Catastrophe_History` (the source tables must already exist).

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
try:
    ORDERS_SCHEMA = dbutils.widgets.get("ORDERS_SCHEMA") or "orders"
except Exception:
    ORDERS_SCHEMA = "orders"

# Bronze hist sources + silver/gold outputs all live in {CATALOG}.orders.
print(f"Source: {CATALOG}.{ORDERS_SCHEMA}.bronze_hist_*")
print(f"Target: {CATALOG}.{ORDERS_SCHEMA}")

In [ ]:
import os
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import pipelines as p

w = WorkspaceClient()

root_abs_path = os.path.abspath("../pipelines/catastrophe_hist")
root_dbx_path = root_abs_path.replace(
    os.environ.get("DATABRICKS_WORKSPACE_ROOT", "/Workspace"),
    "/Workspace"
)

PIPELINE_NAME = f"Catastrophe Historical Aggregation ({CATALOG})"

pipeline_config = dict(
    catalog=CATALOG,
    schema=ORDERS_SCHEMA,
    continuous=False,
    name=PIPELINE_NAME,
    serverless=True,
    configuration={
        "HIST_CATALOG": CATALOG,
        "HIST_SCHEMA": ORDERS_SCHEMA,
    },
    root_path=root_dbx_path,
    libraries=[p.PipelineLibrary(glob=p.PathPattern(include=f"{root_dbx_path}/**"))],
)

existing_pipelines = [
    pl for pl in w.pipelines.list_pipelines(filter=f"name LIKE '{PIPELINE_NAME}'")
    if pl.name == PIPELINE_NAME
]

if existing_pipelines:
    pipeline_id = existing_pipelines[0].pipeline_id
    w.pipelines.update(pipeline_id=pipeline_id, **pipeline_config)
    print(f"♻️ Updated existing pipeline: {pipeline_id}")
else:
    created = w.pipelines.create(**pipeline_config)
    pipeline_id = created.pipeline_id
    import sys
    sys.path.append("../utils")
    from uc_state import add
    add(CATALOG, "pipelines", created)
    print(f"✅ Created pipeline: {pipeline_id}")

In [ ]:
# Historical data is static (generated once by Catastrophe_History). Drop any
# prior silver/gold outputs then full-refresh. Needed because Catalog_Commits
# (or workspace defaults) can leave catalogManaged materialization tables that
# break the next update with DELTA_PATH_BASED_ACCESS_TO_CATALOG_MANAGED_TABLE_BLOCKED.
_PIPELINE_OUTPUTS = [
    "silver_orders_enriched",
    "silver_order_events_latest",
    "gold_orders_by_city_day",
    "gold_orders_by_kitchen_day",
    "gold_disruption_impact",
    "gold_refunds_summary",
    "gold_complaints_summary",
]
for _name in _PIPELINE_OUTPUTS:
    _fq = f"{CATALOG}.{ORDERS_SCHEMA}.{_name}"
    try:
        spark.sql(f"DROP MATERIALIZED VIEW IF EXISTS {_fq}")
        print(f"Dropped MV {_fq}")
    except Exception:
        try:
            spark.sql(f"DROP TABLE IF EXISTS {_fq}")
            print(f"Dropped table {_fq}")
        except Exception as _e:
            print(f"Skip drop {_fq}: {_e}")

update = w.pipelines.start_update(pipeline_id=pipeline_id, full_refresh=True)
print(f"🚀 Started full-refresh pipeline update: {update.update_id}")

while True:
    info = w.pipelines.get(pipeline_id=pipeline_id)
    latest = info.latest_updates[0] if info.latest_updates else None
    state_str = str(latest.state) if latest else "STARTING"
    if "COMPLETED" in state_str:
        print(f"✅ Pipeline finished: {state_str}")
        break
    if "FAILED" in state_str:
        raise RuntimeError(f"Pipeline failed: {state_str}")
    if "CANCELED" in state_str:
        raise RuntimeError(f"Pipeline canceled: {state_str}")
    print(f"  Pipeline state: {state_str}...")
    time.sleep(15)


In [ ]:
print(f"✅ Catastrophe history pipeline stage complete (pipeline_id={pipeline_id})")